# Capítulo 5 — Dicionários: o catálogo da missão

## Missão
Usar dicionários, a estrutura mais usada em Python, para montar o **catálogo de amostras** e a **configuração** do rover. Você vai criar, ler com segurança, percorrer, copiar sem armadilhas, apagar e usar dicionários com várias chaves e aninhados.

**No fim deste caderno você terá:** o módulo `gabarito_rover/catalogo.py`, com o catálogo de amostras indexado por id e agrupado por tipo.

**Leitura essencial (opcional, livro em inglês):**
- Creating a Data Dictionary · PDF p. 192
- Looping through a Dictionary · PDF p. 200
- Data Dictionary Methods · PDF p. 201
- Copying a Dictionary · PDF p. 202
- Having Fun with Multi-Key Dictionaries · PDF p. 206

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados da missão do rover)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_rover/
os.makedirs("../gabarito_rover", exist_ok=True)
open("../gabarito_rover/__init__.py", "a").close()        # gabarito_rover/ é o rover que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_rover")

## Ideia em 30 segundos
Um dicionário liga **chaves** a **valores** e acha qualquer chave instantaneamente.
- `d[chave]` dá erro se a chave não existe; `d.get(chave, padrão)` não dá.
- `items()` percorre os pares.
- Cuidado: `b = a` **não copia**: os dois nomes apontam para o mesmo dicionário.

### 5.1 **Configuração do rover.** Crie o dicionário `config` com `velocidade_max` (cm/s), `bateria_minima`, `camera` (True) e `modo` ("EXPLORAR"). Mude o modo, acrescente `antena`, leia uma chave inexistente com `get` e mostre `len`, `keys` e `values`.

In [ ]:
config = {"velocidade_max": 15, "bateria_minima": 30, "camera": True, "modo": "EXPLORAR"}
config["modo"] = "COLETAR"
config["antena"] = "alto ganho"
print(config.get("laser", "não instalado"), len(config))
print(list(config.keys()), list(config.values()))

### 5.2 **Percorrer.** Percorra `config.items()` mostrando `chave = valor` em ordem alfabética das chaves.

In [ ]:
for chave, valor in sorted(config.items()):
    print(f"{chave:15} = {valor}")

### 5.3 **Métodos úteis.** Com `update`, mude 2 valores de uma vez; com `setdefault`, garanta uma chave `log` com uma lista vazia e acrescente uma mensagem; com `pop`, remova `antena` guardando o valor.

In [ ]:
config.update({"velocidade_max": 10, "camera": False})
config.setdefault("log", []).append("configuração atualizada")
antena = config.pop("antena")
print(antena, config)

### 5.4 **A armadilha da cópia.** Faça `b = config` e mude `b["modo"]`: o que acontece com `config`? Refaça com `config.copy()`. Depois mostre que `copy()` é **raso**: mexer na lista `log` da cópia altera a original. Resolva com `copy.deepcopy`.

In [ ]:
import copy
b = config
b["modo"] = "TESTE"
print(config["modo"])                        # mudou junto: é o mesmo objeto
c = config.copy()
c["log"].append("mexi na cópia")
print(config["log"])                         # a lista interna é compartilhada
d = copy.deepcopy(config)
d["log"].append("só na cópia profunda")
print(config["log"], "|", d["log"])

### 5.5 **Dicionários aninhados e com várias chaves.** Monte `leituras_por_hora`: a chave é a hora (`HH:MM`) e o valor é um dicionário com `temp`, `bateria` e `dist`. Mostre a temperatura das 12:00. Depois crie um dicionário cuja chave é a **tupla** `(linha, coluna)` e cujo valor é a lista de ids das amostras naquela posição.

In [ ]:
leituras_por_hora = {l[0][11:16]: {"temp": l[1], "bateria": l[3], "dist": l[4]} for l in LEITURAS}
print(leituras_por_hora["12:00"]["temp"])
por_pos = {}
for a in AMOSTRAS:
    por_pos.setdefault(a["pos"], []).append(a["id"])
print(por_pos)

### 5.6 **Apagar com segurança.** Remova de `leituras_por_hora` todas as horas com bateria abaixo de 80. Por que não dá para apagar enquanto percorre o próprio dicionário? Percorra uma cópia das chaves (`list(...)`).

In [ ]:
for hora in list(leituras_por_hora):
    if leituras_por_hora[hora]["bateria"] < 80:
        del leituras_por_hora[hora]
print(list(leituras_por_hora))

### 5.7 **Módulo do catálogo.** Escreva `catalogo.py` com:
- `indexar(amostras)`: dicionário id → amostra;
- `agrupar_por_tipo(amostras)`: tipo → lista de ids;
- `resumo(amostras)`: tipo → `{"quantidade": n, "massa_total_g": soma}`.

_A célula grava `gabarito_rover/catalogo.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_rover/catalogo.py
"""Catálogo de amostras coletadas pelo rover."""


def indexar(amostras: list[dict]) -> dict:
    return {a["id"]: a for a in amostras}


def agrupar_por_tipo(amostras: list[dict]) -> dict:
    grupos = {}
    for a in amostras:
        grupos.setdefault(a["tipo"], []).append(a["id"])
    return grupos


def resumo(amostras: list[dict]) -> dict:
    saida = {}
    for a in amostras:
        r = saida.setdefault(a["tipo"], {"quantidade": 0, "massa_total_g": 0.0})
        r["quantidade"] += 1
        r["massa_total_g"] = round(r["massa_total_g"] + a["massa_g"], 1)
    return saida


### 5.8 **Teste o catálogo.** Mostre a amostra `AM-003` pelo índice, os grupos por tipo e o resumo.

In [ ]:
from gabarito_rover.catalogo import indexar, agrupar_por_tipo, resumo
print(indexar(AMOSTRAS)["AM-003"])
print(agrupar_por_tipo(AMOSTRAS))
resumo(AMOSTRAS)

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
